<a href="https://colab.research.google.com/github/manyajain2435/AAI_primer/blob/main/LLM_Session_1_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Tool Usage & The ReAct Loop (Reason + Act) - The Heart of Agency

Single tool call to multi-step reasoning.

In [1]:
# run This cell if you are in Google Colab

import os, datetime, shutil
from google.colab import drive

# Mount Drive
# drive.mount('/content/drive', force_remount=True)


In [6]:
!pip install -q groq

import json
from google.colab import userdata
from groq import Groq

# Get API key from Colab Secrets
api_key = userdata.get("gsk_WORKSHOP_KEY")

# Create Groq client
client = Groq(api_key=api_key)

def ask_llm(question):
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "user", "content": question}
            ]
        )

        return response.choices[0].message.content

    except Exception as e:
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Message: {str(e)}")
        return None


# Test the connection
result = ask_llm("What's the capital of France?")

if result:
    print("SUCCESS:", result)

SUCCESS: The capital of France is Paris.


### Why Tool usage or Function support is needed ?

In [7]:
# Test
result = ask_llm("Whats the Weather in Bangalore ?'")
if result:
    print(f"SUCCESS: {result}")

SUCCESS: However, I'm a large language model, I don't have real-time access to current weather conditions. I can suggest some options to help you find the current weather in Bangalore:

1. **Check online weather websites**: You can check websites like AccuWeather, Weather.com, or the Indian Meteorological Department (IMD) website for current weather conditions in Bangalore.
2. **Use a weather app**: You can download a weather app like Dark Sky, Weather Underground, or the IMD app on your smartphone to get the current weather conditions in Bangalore.
3. **Search online**: You can simply type "current weather in Bangalore" or "weather in Bangalore today" on a search engine like Google to get the latest information.

As per my training data (which is up to 2023), Bangalore's climate is generally warm and humid, with three main seasons:

- Summer (March to May): Hot and dry
- Monsoon (June to September): Wet and humid
- Winter (October to February): Mild and pleasant

Please note that the 

### Giving the Agent Hands : tool Usage
Introducing the Function Definition (JSON Schema).
"This is the instruction manual we give the LLM so

In [8]:
# Tool Definitions
import json

# 1. Define the Python Tool
def get_weather(city: str):
    # Mock database for workshop
    mock_db = {"mumbai": "32C, Humid", "delhi": "28C, Smoggy", "bangalore": "27C, Sultry"}
    return mock_db.get(city.lower(), "Data not available")

# 2. Write the TOOL SCHEMA (This is the tricky part students learn)
tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# 3. The AGENTIC CALL
def agent_ask(user_query):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": user_query}],
        tools=tool_schema, # <--- THE MAGIC LINE
        tool_choice="auto"
    )
    return response


### The ReAct Loop (Reason + Act) - The Heart of Agency

Goal: Move from single tool call to multi-step reasoning.

In [9]:

def run_agent(user_query):
    messages = [{"role": "user", "content": user_query}]

    # The Loop - Students fill in the condition
    while True:  # <--- STUDENTS FILL THIS
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=tool_schema,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        messages.append(response_message) # Append assistant thought

        # Check if LLM wants to call a tool
        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                # EXECUTE THE PYTHON FUNCTION (Student fills this)
                if function_name == "get_weather":
                    result = get_weather(function_args.get("city"))
                    # Feed result back to LLM
                    messages.append({
                        "role": "tool",
                        "content": result,
                        "tool_call_id": tool_call.id
                    })
        else:
            # NO MORE TOOLS -> ANSWER IS READY
            return response_message.content



In [10]:
print(run_agent("Is it warmer in Mumbai or Delhi right now?"))

Based on the information provided, it appears to be warmer in Mumbai (32C) compared to Delhi (28C) at this time. However, the humidity in Mumbai and the smog levels in Delhi might affect how warm it feels.
